# Paper 2 follow-up experiments: Colab T4 (cell B: LR x decay grid; cell G: ResNet-18 on CIFAR-100)

1. **E8** (cell B, ~2 h): the missing points of the 6 LR x 6 decay-coefficient grid for Adam and AdamW (1 seed, 50 epochs), used for the hyperparameter-robustness analysis (heatmap).
2. **E9** (cell G, ~22 T4 hours, several sessions): the six 2x2 configurations on ResNet-18 + CIFAR-100: 30-epoch LR search, then 5 seeds x 100 epochs.

- Code: upload a zip of this repository as `adam-bench-v5.zip` to `MyDrive` (the notebook unpacks it).
- Results are written directly to `MyDrive/adam-bench-v5/results_exp`, so a lost session loses no data.
- Runtime -> Change runtime type -> **T4 GPU**.
- After a disconnect: reopen and **Runtime -> Run all**. Finished runs are skipped; an interrupted run resumes from its checkpoint.
- To run E8 only: set `EXPS = ['E8']`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/adam-bench-v5'   # change if stored elsewhere
RESULTS = f'{PROJECT}/results_exp'
WORK = '/content/adam-bench-v5'
DATA = '/content/data'
CELLS = ['B', 'G']
EXPS = ['E8', 'E9']                  # priority order: E8 (cell B) first, then E9 (cell G)
RUN_SELFTEST = True                  # True the first time; set False after the self-test passes

import os, zipfile
ZIP = '/content/drive/MyDrive/adam-bench-v5.zip'
if os.path.exists(ZIP):   # always unpack the latest code (the zip holds no results, so nothing is overwritten)
    zipfile.ZipFile(ZIP).extractall('/content/drive/MyDrive')
    print('Unpacked', ZIP)
assert os.path.exists(f'{PROJECT}/run_exp.py'), f'Code not found in {PROJECT}'
!rsync -a --delete --exclude results_exp --exclude data {PROJECT}/ {WORK}/
%cd {WORK}
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# === Pin the torch + cuDNN versions of the original benchmark (the runner is a subprocess, so no restart is needed) ===
import subprocess, sys
WANT = {'B': ('2.11.0', '0.26.0', 'cu130', 92700),
        'C': ('2.11.0', '0.26.0', 'cu128', 91900),
        'D': ('2.11.0', '0.26.0', 'cu128', 91900),
        'E': ('2.11.0', '0.26.0', 'cu128', 91900),
        'F': ('2.11.0', '0.26.0', 'cu128', 91900),
        'H': ('2.11.0', '0.26.0', 'cu128', 91900),
        'G': ('2.11.0', '0.26.0', 'cu130', 92700)}
tv, vv, cu, cudnn_want = WANT[CELLS[0]]
want = f'{tv}+{cu}'

def probe():
    out = subprocess.run([sys.executable, '-c',
                          'import torch; print(torch.__version__); print(torch.backends.cudnn.version()); '
                          'print("|".join(torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())))'],
                         capture_output=True, text=True).stdout.splitlines()
    return out[0], int(out[1]), out[2] if len(out) > 2 else ''

have, cudnn, gpus = probe()
print('torch installed:', have, '| required:', want)
if have != want:
    print('Reinstalling torch to match the original benchmark (a few minutes)...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           f'torch=={tv}+{cu}', f'torchvision=={vv}+{cu}',
                           '--index-url', f'https://download.pytorch.org/whl/{cu}'])
    have, cudnn, gpus = probe()
    print('torch after install:', have)
assert have == want, f'Could not install torch {want}; stopping.'

if cudnn != cudnn_want:
    pkg = 'nvidia-cudnn-cu13' if cu.startswith('cu13') else 'nvidia-cudnn-cu12'
    spec = f'{pkg}=={cudnn_want // 10000}.{cudnn_want // 100 % 100}.*'
    print(f'cuDNN is {cudnn}, required {cudnn_want} -> installing {spec}')
    subprocess.call([sys.executable, '-m', 'pip', 'install', '-q', spec])
    have, cudnn, gpus = probe()
print('torch', have, '| cuDNN', cudnn, '(required', cudnn_want, ') | GPU:', gpus)
assert 'T4' in gpus, 'Not a T4 GPU: switch the runtime/accelerator to T4.'
FORCE_ENV = cudnn != cudnn_want
if FORCE_ENV:
    print('WARNING: cuDNN could not be matched. Continuing with --force-env: torch and GPU match, '
          'only the cuDNN patch level differs; results are statistically equivalent but may not be bit-identical.')
else:
    print('Environment matches the original benchmark.')

In [ ]:
# === Self-test (first time, ~2 min): it must report success ===
if RUN_SELFTEST:
    assert subprocess.call([sys.executable, 'run_exp.py', '--selftest', '--out', '/content/tmp_selftest']) == 0, 'SELFTEST failed'

In [ ]:
# === Run ===
import subprocess, sys
subprocess.call([sys.executable, 'run_exp.py', '--plan', '--cells', *CELLS, '--exp', *EXPS])
p = subprocess.Popen([sys.executable, '-u', 'run_exp.py', '--exp', *EXPS, '--cells', *CELLS,
                      '--out', RESULTS, '--data', DATA] + (['--force-env'] if FORCE_ENV else []),
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end='', flush=True)
rc = p.wait()
print('Exit code', rc, '(2 = environment/configuration error, see the messages above)')

In [ ]:
# === Progress and packaging for download ===
import os, shutil, subprocess, sys
subprocess.call([sys.executable, 'run_exp.py', '--status', '--cells', *CELLS, '--exp', *EXPS, '--out', RESULTS])
if os.path.isdir(RESULTS):
    z = shutil.make_archive(f'{PROJECT}/results_exp_v5_colab', 'zip', PROJECT, 'results_exp')
    print('Created', z)